In [1]:
import pandas as pd
import numpy as np

df = pd.read_stata("WIOT2014_October16_ROW.dta")

df = df[df['Country']!='TOT']

df = df.groupby("Country").sum().reset_index()

for country in list(df['Country']):
    df[country] = df[[col for col in df.columns if col.startswith("v" + country)]].sum(axis=1)

df = df[["Country"] + list(df['Country'])]

In [2]:
df2 = df.iloc[:,~(df.columns.isin(['USA','CHN','CAN','MEX']))]
df3 = df2['Country'].copy().to_frame()
df['ROW'] = np.sum(df2.iloc[:,1:],axis = 1)

In [3]:
exclude = ["USA", "CHN", "CAN", "MEX"]
subset = df.loc[~df["Country"].isin(exclude)]
new_row = subset.sum(axis=0)        # Series: sums of each column

# Turn the Series into a 1-row DataFrame
agg_df = new_row.to_frame().T      # or: pd.DataFrame([new_row])

# Ensure the columns are in the same order as df
agg_df = agg_df[df.columns]

# Overwrite the first column (country code/name) with 'AGG_OTHERS'
country_col = df.columns[0]        # e.g. "Country"
agg_df[country_col] = "AGG_OTHERS"

In [4]:
us_2014 = {"USA": [
    df[df["Country"] == "USA"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CHN"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "MEX"]["USA"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CAN"]["USA"].values[0] / df["USA"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["USA"].values[0] / df["USA"].sum()
],
     "CHN": [
    df[df["Country"] == "USA"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "CHN"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "MEX"]["CHN"].values[0] / df["CHN"].sum(),
    df[df["Country"] == "CAN"]["CHN"].values[0] / df["CHN"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["CHN"].values[0] / df["CHN"].sum()
],
     "MEX": [
    df[df["Country"] == "USA"]["MEX"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CHN"]["MEX"].values[0] / df["MEX"].sum(),
    df[df["Country"] == "MEX"]["MEX"].values[0] / df["MEX"].sum(),
    df[df["Country"] == "CAN"]["MEX"].values[0] / df["MEX"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["MEX"].values[0] / df["MEX"].sum()
],
     "CAN": [
    df[df["Country"] == "USA"]["CAN"].values[0] / df["USA"].sum(),
    df[df["Country"] == "CHN"]["CAN"].values[0] / df["CAN"].sum(),
    df[df["Country"] == "MEX"]["CAN"].values[0] / df["CAN"].sum(),
    df[df["Country"] == "CAN"]["CAN"].values[0] / df["CAN"].sum(),
    agg_df[agg_df["Country"] == "AGG_OTHERS"]["CAN"].values[0] / df["CAN"].sum()
]}

us_2000 = np.array([[0.932,0.003,0.007,0.012,0.046],
                [0.004,0.931,0.000,0.001,0.064],
                [0.097,0.002,0.845,0.003,0.054],
                [0.114,0.004,0.005,0.816,0.061]])

df_l2000 = pd.DataFrame(us_2000, index=["USA", "CHN", "MEX", "CAN"], columns=["USA", "CHN", "MEX", "CAN", "ROW"])

df_l2014 = pd.DataFrame(us_2014, index=["USA", "CHN", "MEX", "CAN", "ROW"]).T

In [20]:
df_l2000

,USA,CHN,MEX,CAN,ROW
USA,0.932,0.003,0.007,0.012,0.046
CHN,0.004,0.931,0.000,0.001,0.064
MEX,0.097,0.002,0.845,0.003,0.054
CAN,0.114,0.004,0.005,0.816,0.061


In [21]:
df_l2014

,USA,CHN,MEX,CAN,ROW
USA,0.923448,0.011043,0.008501,0.011144,0.045864
CHN,0.003596,0.940864,0.000237,0.000572,0.054732
MEX,0.005678,0.018048,0.829805,0.003879,0.064178
CAN,0.009282,0.015352,0.006111,0.831556,0.056693


In [9]:
eps = 5

lambda_US_2000 = df_l2000.loc["USA", "USA"]
lambda_US_2014 = df_l2014.loc["USA", "USA"]

g_us_2000 = lambda_US_2000 ** (-1/eps)
g_us_2014 = lambda_US_2014 ** (-1/eps)

g_dif_us = np.log(g_us_2014) - np.log(g_us_2000)

lambda_CHN_2000 = df_l2000.loc["CHN", "CHN"]
lambda_CHN_2014 = df_l2014.loc["CHN", "CHN"]

g_chn_2000 = lambda_CHN_2000 ** (-1/eps)
g_chn_2014 = lambda_CHN_2014 ** (-1/eps)

g_dif_chn = np.log(g_chn_2014) - np.log(g_chn_2000)

lambda_MEX_2000 = df_l2000.loc["MEX", "MEX"]
lambda_MEX_2014 = df_l2014.loc["MEX", "MEX"]

g_mex_2000 = lambda_MEX_2000 ** (-1/eps)
g_mex_2014 = lambda_MEX_2014 ** (-1/eps)

g_dif_mex = np.log(g_mex_2014) - np.log(g_mex_2000)

lambda_CAN_2000 = df_l2000.loc["CAN", "CAN"]
lambda_CAN_2014 = df_l2014.loc["CAN", "CAN"]

g_can_2000 = lambda_CAN_2000 ** (-1/eps)
g_can_2014 = lambda_CAN_2014 ** (-1/eps)

g_dif_can = np.log(g_can_2014) - np.log(g_can_2000)

g_diffs = {
    "USA": g_dif_us,
    "CAN": g_dif_can,
    "MEX": g_dif_mex,
    "CHN": g_dif_chn
}

pd.DataFrame(g_diffs, index=["Growth Difference"]).T

,Growth Difference
USA,0.001844
CAN,-0.003777
MEX,0.003629
CHN,-0.002108


![image](formula.png)

In [13]:
df_l2000.loc["CAN","USA"]

np.float64(0.114)

In [16]:
1/eps * df_l2000.loc["CAN","USA"] / df_l2000.loc["CAN","CAN"] * (df_l2014.loc["CAN", "USA"] - df_l2000.loc["CAN", "USA"]) / df_l2000.loc["CAN","USA"]

np.float64(-0.025666207268604625)

In [ ]:
e = 5.0

l2014 = df_l2014.to_numpy()

l2000 = np.array([[0.932, 0.003,0.007,0.012,0.046],
                [0.004,0.931,0.0000001,0.001,0.064],
                [0.097,0.002,0.845,0.003,0.054],
                [0.114,0.004,0.005,0.816,0.061]])

t7 = np.full(shape=(4,6),fill_value=np.nan)

for i in range(0,4):
    arr = np.concatenate([np.arange(0, i), np.arange(i + 1, 5)])
    t7[i,0] = 1/e*np.sum(l2000[i,arr]/l2000[i,i]*(l2014[i,arr]-l2000[i,arr])/l2000[i,arr])

for i in range(0,4):
    for j in range(0,5):
            if j == i:
                 continue
            t7[i,j+1] = 1/e*l2000[i,j]/l2000[i,i]*(l2014[i,j]-l2000[i,j])/l2000[i,j]/t7[i,0]

df7 = pd.DataFrame(t7,columns=['GFT','US Cont','CHN Cont','MEX Cont','CAN Cont','ROW Cont'])
df7

,GFT,US Cont,CHN Cont,MEX Cont,CAN Cont,ROW Cont
0,0.001835,NaN,0.940500,0.175507,-0.100153,-0.015855
1,-0.002119,0.040987,NaN,-0.024049,0.043423,0.939639
2,-0.015199,1.422076,-0.249902,NaN,-0.013681,-0.158493
3,-0.023667,1.084458,-0.117557,-0.011501,NaN,0.044600
